# EYEWAZ — train your Urdu Piper voice (piper1-gpl)

Uses the **maintained** Piper trainer — installs on Colab's normal Python, GPU,
no version hacks. Steps: Runtime → Change runtime type → **GPU (T4)**; upload
`dataset-<VOICE>.zip` via the 📁 Files panel; Runtime → **Run all**.


### 1. GPU on?


In [ ]:
!nvidia-smi -L || print('NO GPU — Runtime > Change runtime type > GPU first!')


### 2. Install Piper trainer (+ espeak-ng)


In [ ]:
!apt-get -qq install -y espeak-ng build-essential cmake ninja-build >/dev/null
import os
if not os.path.exists('/content/piper1'):
    !git clone -q https://github.com/OHF-Voice/piper1-gpl /content/piper1
%cd /content/piper1
!pip install -q -e '.[train]'
!bash build_monotonic_align.sh
print('Piper trainer installed ✓')


### 3. Your dataset
Upload `dataset-<VOICE>.zip` via the 📁 Files panel (drag it to the top level),
then run this. Builds the `wav|text` CSV piper1-gpl expects.


In [ ]:
import os, glob
os.chdir('/content')
VOICE = 'male'
zip_path = f'/content/dataset-{VOICE}.zip'
if not os.path.exists(zip_path):
    hits = glob.glob('/content/**/dataset-*.zip', recursive=True)
    if hits: os.replace(hits[0], zip_path)
assert os.path.exists(zip_path), f'Upload dataset-{VOICE}.zip via the Files panel, then re-run.'
!unzip -oq {zip_path} -d /content/
AUDIO = f'/content/dataset-{VOICE}/wav'
CSV = f'/content/{VOICE}.csv'
with open(f'/content/dataset-{VOICE}/metadata.csv', encoding='utf-8') as f, open(CSV,'w',encoding='utf-8') as o:
    for line in f:
        line=line.strip()
        if '|' in line:
            cid,text=line.split('|',1); o.write(f'{cid}.wav|{text}\n')
print('clips:', len(os.listdir(AUDIO)), '| csv:', CSV)


### 4. Pretrained checkpoint to fine-tune from
Speeds up training a lot even though it's a different language.


In [ ]:
CKPT='https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
!wget -q --show-progress -O /content/pretrained.ckpt "$CKPT"
import os; print('checkpoint MB:', round(os.path.getsize('/content/pretrained.ckpt')/1e6,1))


### 5. Train
Logs progress; checkpoints are written under `lightning_logs/`. With ~6 min of
data this is a **pilot**. **Stop the cell** (■) when it's trained enough, then run cell 6.


In [ ]:
%cd /content/piper1
!python3 -m piper.train fit \
  --data.voice_name eyewaz-urdu-{VOICE} \
  --data.csv_path /content/{VOICE}.csv \
  --data.audio_dir /content/dataset-{VOICE}/wav \
  --model.sample_rate 22050 \
  --data.espeak_voice ur \
  --data.cache_dir /content/cache \
  --data.config_path /content/eyewaz-urdu-{VOICE}.json \
  --data.batch_size 16 \
  --ckpt_path /content/pretrained.ckpt \
  --trainer.max_epochs 1000 \
  --trainer.accelerator gpu --trainer.devices 1


### 6. Export to ONNX + download


In [ ]:
import glob, os
cks = sorted(glob.glob('/content/piper1/lightning_logs/version_*/checkpoints/*.ckpt'), key=os.path.getmtime)
assert cks, 'No checkpoint yet — let cell 5 train longer.'
ck = cks[-1]; print('exporting', ck)
!python3 -m piper.train.export_onnx --checkpoint {ck} --output-file /content/eyewaz-urdu-{VOICE}.onnx
!cp /content/eyewaz-urdu-{VOICE}.json /content/eyewaz-urdu-{VOICE}.onnx.json
from google.colab import files
files.download(f'/content/eyewaz-urdu-{VOICE}.onnx')
files.download(f'/content/eyewaz-urdu-{VOICE}.onnx.json')


### Done
Send me `eyewaz-urdu-male.onnx` + `.onnx.json` and I'll wire your voice into every EYEWAZ surface.
